# 3D Chebyshev interpolation

The same tensor-grid workflow in three dimensions, plus two features that
matter as dimension grows:

- **pruning** (`optimize`) trims trailing coefficient slices along every axis
  under a global error budget, and
- **precomputation** (`precompute_x`) collapses the leading axis at a fixed
  value, making repeated evaluations over the remaining variables much
  cheaper.

In [1]:
import numpy as np
from time import perf_counter

from chebyshevnd import ChebyshevInterpolant3D, chebyshev_nodes

In [2]:
f = lambda x, y, z: np.sin(x) * np.cos(y) * np.exp(-z)
domains = [(0.0, 2.0), (-1.0, 3.0), (1.0, 4.0)]
Ns = (20, 20, 20)

axes = [chebyshev_nodes(n, a, b) for n, (a, b) in zip(Ns, domains)]
grids = np.meshgrid(*axes, indexing="ij")
interp = ChebyshevInterpolant3D(f(*grids),
                                *domains[0], *domains[1], *domains[2])

In [3]:
# Check the error at random points in the domain
rng = np.random.default_rng(0)
points = [tuple(rng.uniform(a, b) for a, b in domains) for _ in range(200)]
err = max(abs(interp.evaluate(*p) - f(*p)) for p in points)
print(f"max error at 200 random points: {err:.3e}")
print(f"estimated absolute error:       {interp.absolute_error():.3e}")

max error at 200 random points: 1.388e-16
estimated absolute error:       2.761e-16


## Pruning the coefficient tensor

`optimize` greedily removes the cheapest trailing slices (in the L1 sense)
along any axis until the error budget `rel_tol` is exhausted. The kept block
is reported by `effective_grid()`.

In [4]:
print(f"full grid:      {Ns}")
interp.optimize(rel_tol=1e-8)
print(f"effective grid: {interp.effective_grid()}")

err_opt = max(abs(interp.evaluate(*p) - f(*p)) for p in points)
print(f"max error after pruning: {err_opt:.3e}")

full grid:      (20, 20, 20)
effective grid: (10, 13, 11)
max error after pruning: 2.891e-10


## Precomputing a dimension

When one variable is held fixed across many evaluations — sweeping a
parameter, scanning a slice — collapse it once with `precompute_x` and then
call `evaluate_yz`, which only does 2D work.

In [5]:
x0 = 1.3
interp.precompute_x(x0)

# Consistency check
y0, z0 = 0.5, 2.5
print(f"evaluate:    {interp.evaluate(x0, y0, z0):.12f}")
print(f"evaluate_yz: {interp.evaluate_yz(y0, z0):.12f}")

evaluate:    0.069411227614


evaluate_yz: 0.069411227582


In [6]:
# Timing: full 3D evaluation vs. 2D evaluation with x collapsed
yz_points = [(rng.uniform(*domains[1]), rng.uniform(*domains[2])) for _ in range(2000)]

t0 = perf_counter()
for y, z in yz_points:
    interp.evaluate(x0, y, z)
t_full = perf_counter() - t0

t0 = perf_counter()
for y, z in yz_points:
    interp.evaluate_yz(y, z)
t_pre = perf_counter() - t0

print(f"2000 evaluations, full 3D:        {t_full*1e3:7.1f} ms")
print(f"2000 evaluations, precomputed x:  {t_pre*1e3:7.1f} ms")
print(f"speed-up: {t_full/t_pre:.1f}x")

2000 evaluations, full 3D:            3.4 ms
2000 evaluations, precomputed x:      1.2 ms
speed-up: 2.8x
